In [5]:
(import std.causal.render)
(import std.io)
(import std.jupyter)

;; Validates: every edge well-formed AND directed sub-graph is acyclic.
;; Returns a normalised edge list you can pass to any renderer.
(define energy-dag
  (define-dag
    '((oil   ->  XLE)
      (oil   ->  USO)
      (oil   ->  SPY)
      (rates ->  SHY)
      (rates ->  IEF)
      (fx    ->  VGK)
      (XLE   <-> USO))))      ;; latent confounder between energy names

energy-dag    ;; cell value: prints normalised edges

(define dot
  (dag:->dot energy-dag
             '((title   . "Energy-shock SCM")
               (rankdir . "TB"))))      ;; top-to-bottom

(display dot)
;; Persist to disk, then run `dot -Tsvg` outside the notebook.
;;(call-with-output-file "energy-dag.dot"
;;  (lambda (p) (display dot p)))

digraph "Energy-shock SCM" {
  rankdir=TB;
  "USO" -> "XLE" [dir=both, style=dashed];
  "fx" -> "VGK";
  "oil" -> "SPY";
  "oil" -> "USO";
  "oil" -> "XLE";
  "rates" -> "IEF";
  "rates" -> "SHY";
}

In [6]:
(import std.causal.render)
(import std.jupyter)

(define prior-dag
  (define-dag '((oil -> XLE) (oil -> USO) (rates -> IEF) (fx -> VGK))))

(define learned-dag
  (define-dag '((oil -> XLE) (oil -> USO) (oil -> SPY)
                (rates -> SHY) (rates -> IEF) (fx -> VGK)
                (XLE <-> USO))))

(jupyter:markdown
  (string-append
    "### Prior\n\n```mermaid\n" (dag:->mermaid prior-dag)  "\n```\n\n"
    "### Learned (NOTEARS)\n\n```mermaid\n" (dag:->mermaid learned-dag) "\n```\n"))

### Prior

```mermaid
flowchart LR
  n0["IEF"]
  n1["USO"]
  n2["VGK"]
  n3["XLE"]
  n4["fx"]
  n5["oil"]
  n6["rates"]
  n4 --> n2
  n5 --> n1
  n5 --> n3
  n6 --> n0

```

### Learned (NOTEARS)

```mermaid
flowchart LR
  n0["IEF"]
  n1["SHY"]
  n2["SPY"]
  n3["USO"]
  n4["VGK"]
  n5["XLE"]
  n6["fx"]
  n7["oil"]
  n8["rates"]
  n3 -.-> n5
  n5 -.-> n3
  n6 --> n4
  n7 --> n2
  n7 --> n3
  n7 --> n5
  n8 --> n0
  n8 --> n1

```
